In [42]:
import os
import torch
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import DataLoader,random_split

from model_tools import load_configs, add_config
from data_utils import SimulationData,generate_input_filename
from VAE import VAE, EmbeddedDataset, save_model, get_scheduler, reconstruct_z2fromz1

Modifica per caricare i dati di input e dare un nome al modello

In [43]:
tau = 15
embedding_dim = 20
alpha = 6
beta = 0
simulation_name= generate_input_filename(tau, embedding_dim, alpha, beta)
model_name = "Seconda"
scheduler_name = "ReduceLROnPlateau"
print(simulation_name)

delay15dim20_noise6,0


In [44]:
# Percorso del database
model_database_path = "modelDatabase.json"

# Carica tutte le configurazioni disponibili
all_configs = load_configs(model_database_path)

print(model_name)

Seconda


Modifica per cambiare la configurazione della rete: latent dim e layer (insieme), occhio agli input

In [45]:
# Se vuoi creare una nuova configurazione
new_config = {
    "latent_dim": 5,
    "encoder_layers": [20,17,15,12,10,8,5,3],
    "decoder_layers": [3,7,10,15,20,25,30,35,40],
    "beta_method": "sigmoid", #constant, sigmoid, linear, linear_decay, exponential_decay
    "beta_value": 0.5,
    "kl_annealing_epochs": 100, # Epoche di warmup
    "decay_start": 100, # Epoch di inizio del decadimento
    "decay_epoch": 50, # Durata del decadimento
    "early_stopping": {"patience": 10, "delta": -1}, # Early stopping -1 per disattivare
    "dataset": simulation_name,
    "batch_size": 32,
    "optimizer": "Adam",
    "learning_rate": 0.01,
    "scheduler": {
        "type": scheduler_name,  # Scheduler alternativo
        "params": {
            "mode": "min",  # Monitorare la perdita minima
            "factor": 0.1,
            "patience": 5
        }
    }
}

epochs = 150

# Aggiungi la nuova configurazione al database
add_config(new_config, model_name, model_database_path)
print(all_configs)

{}


Modifica il NOME del modello da ALLENARE

In [46]:
# Carica una configurazione esistente
selected_model = new_config

# Carica i dati di simulazione
simulation = SimulationData.load_from_params(
    tau=tau, embedding_dim=embedding_dim, alpha=alpha, beta=beta, input_path="Simulation data"
)

# Seleziona gli embedding dal SimulationData
input_variable = "y1_embedding"  # Embedding di y1
output_variable1 = "y2_embedding_1"  # Embedding di y2_1
output_variable2 = "y2_embedding_2"  # Embedding di y2_2

# Recupera gli embedding da simulation
embedding_y1 = getattr(simulation, input_variable)
embedding_y2_1 = getattr(simulation, output_variable1)
embedding_y2_2 = getattr(simulation, output_variable2)

# Verifica che gli embedding siano stati generati correttamente
if embedding_y1 is None or embedding_y2_1 is None or embedding_y2_2 is None:
    raise ValueError("Uno o più embedding non sono stati generati correttamente!")

# Crea il dataset
embedded_dataset = EmbeddedDataset(embedding_y1, embedding_y2_1, embedding_y2_2)


print("y1_embedding shape:", simulation.y1_embedding.shape)  # Deve essere (n_samples, embedding_dim)
print(f"Dimensione embedding_y2_1: {embedding_y2_1.shape}")
print(f"Dimensione embedding_y2_2: {embedding_y2_2.shape}")


y1_embedding shape: (9715, 20)
Dimensione embedding_y2_1: (9715, 20)
Dimensione embedding_y2_2: (9715, 20)


In [47]:

# Suddivisione del dataset
total_size = len(embedding_y1)
train_size = int(0.60 * total_size)  # 60% per il training
val_size = total_size - train_size   # 40% per la validazione

# Suddivisione del dataset senza random split
train_data = torch.utils.data.Subset(embedded_dataset, range(train_size))
val_data = torch.utils.data.Subset(embedded_dataset, range(train_size, total_size))

# Stampa delle dimensioni
print(f"Dimensione del dataset: {total_size}")
print(f"Numero di campioni di training: {len(train_data)}")
print(f"Numero di campioni di validazione: {len(val_data)}")

# Prepara il DataLoader
batch_size = selected_model["batch_size"]
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

# Debug del batch
for batch in train_loader:
    x, y = batch  # Decomponi input e output
    x = x.float()
    y = y.float()
    print("Shape of input:", x.shape)
    break

Dimensione del dataset: 9715
Numero di campioni di training: 5829
Numero di campioni di validazione: 3886
Shape of input: torch.Size([32, 20])


Occhio al Learning rate: 0.1 e 0.01 causano instabilità numerica molto alte nei training, con conseguenti comportamenti anomali

In [48]:
# Crea il modello
encoder_layers = selected_model["encoder_layers"]
decoder_layers = selected_model["decoder_layers"]
vae_model = VAE(encoder_layers,decoder_layers)

# Configura ottimizzatore e scheduler
optimizer = optim.Adam(vae_model.parameters(), lr=selected_model["learning_rate"])

# Configura lo scheduler
scheduler_config = selected_model.get("scheduler")
scheduler = get_scheduler(optimizer, scheduler_config)

# Allenamento del modello
train_losses, val_losses, recon_losses, kld_losses, beta_values, stopped_epoch, gradient_history, lr_evolution = reconstruct_z2fromz1(
    epochs=epochs,
    train_loader=train_loader,
    val_loader=val_loader,  # Aggiungi se hai un dataset di validazione
    model=vae_model,
    optimizer=optimizer,
    scheduler=scheduler,
    scheduler_config=scheduler_config,
    kl_annealing_epochs=selected_model["kl_annealing_epochs"],
    decay_epoch=selected_model["decay_epoch"],
    decay_start=selected_model["decay_start"],
    beta_method=selected_model["beta_method"],
    beta_value=selected_model["beta_value"],
    early_stopping_params=selected_model["early_stopping"]
)


Epoch [1/150], Rec Loss: 1.4011, KLD Loss: 0.0139, Beta: 0.0067, Training Loss: 1.4012
Epoch [2/150], Rec Loss: 1.3990, KLD Loss: 0.0000, Beta: 0.0074, Training Loss: 1.3990
Epoch [3/150], Rec Loss: 1.3995, KLD Loss: 0.0000, Beta: 0.0082, Training Loss: 1.3995
Epoch [4/150], Rec Loss: 1.3996, KLD Loss: 0.0000, Beta: 0.0090, Training Loss: 1.3996
Epoch [5/150], Rec Loss: 1.3994, KLD Loss: 0.0000, Beta: 0.0100, Training Loss: 1.3994
Epoch [6/150], Rec Loss: 1.3993, KLD Loss: 0.0000, Beta: 0.0110, Training Loss: 1.3993
Epoch [7/150], Rec Loss: 1.3996, KLD Loss: 0.0000, Beta: 0.0121, Training Loss: 1.3996
Epoch [8/150], Rec Loss: 1.3992, KLD Loss: 0.0000, Beta: 0.0134, Training Loss: 1.3992
Epoch [9/150], Rec Loss: 1.3997, KLD Loss: 0.0000, Beta: 0.0148, Training Loss: 1.3997
Epoch [10/150], Rec Loss: 1.3997, KLD Loss: 0.0000, Beta: 0.0163, Training Loss: 1.3997
Epoch [11/150], Rec Loss: 1.3995, KLD Loss: 0.0000, Beta: 0.0180, Training Loss: 1.3995
Epoch [12/150], Rec Loss: 1.3984, KLD Los

KeyboardInterrupt: 

In [ ]:
# Carica i gradienti salvati
epochs = range(len(gradient_history["encoder"]))

plt.figure(figsize=(10, 6))
plt.plot(epochs, gradient_history["encoder"], label="Encoder Gradient Norm")
#plt.plot(epochs, gradient_history["decoder"], label="Decoder Gradient Norm")
plt.plot(epochs, gradient_history["latent"], label="Latent Gradient Norm")
plt.xlabel("Epoch")
plt.ylabel("Gradient Norm")
plt.title("Gradient Norm Evolution During Training")
plt.legend()
plt.grid()
plt.show()

In [ ]:
def visualize_latent_space_with_pca(model, dataloader, device='cpu', n_components=2):
    model.eval()
    model.to(device)
    
    latent_mu = []
    true_labels = []

    # Estrazione dello spazio latente
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            _, mu, _ = model(inputs)  # Ottieni solo μ dallo spazio latente
            latent_mu.append(mu.cpu().numpy())
            true_labels.append(labels.cpu().numpy())

    # Concatena correttamente tutti i batch
    latent_mu = np.vstack(latent_mu)
    true_labels = np.concatenate(true_labels)[:len(latent_mu)]  # Allinea la dimensione a latent_mu

    # Applica la PCA per ridurre lo spazio latente
    pca = PCA(n_components=n_components)
    latent_pca = pca.fit_transform(latent_mu)

    # Visualizza il risultato
    if n_components == 2:
        plt.figure(figsize=(8, 6))
        scatter = plt.scatter(latent_pca[:, 0], latent_pca[:, 1], 
                               c=true_labels[:len(latent_pca)], cmap='viridis', s=10)
        plt.colorbar(scatter, label='Class Labels')
        plt.title('Latent Space Visualization with PCA (2D)')
        plt.xlabel('Principal Component 1')
        plt.ylabel('Principal Component 2')
        plt.show()
    elif n_components == 3:
        from mpl_toolkits.mplot3d import Axes3D
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        scatter = ax.scatter(latent_pca[:, 0], latent_pca[:, 1], latent_pca[:, 2],
                              c=true_labels[:len(latent_pca)], cmap='viridis', s=10)
        fig.colorbar(scatter, label='Class Labels')
        ax.set_title('Latent Space Visualization with PCA (3D)')
        ax.set_xlabel('Principal Component 1')
        ax.set_ylabel('Principal Component 2')
        ax.set_zlabel('Principal Component 3')
        plt.show()
    else:
        print(f"PCA visualization not supported for {n_components} dimensions.")

        


In [ ]:
# Esegui la visualizzazione
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
visualize_latent_space_with_pca(vae_model, val_data, device=device, n_components=2)

In [ ]:
# Carica la configurazione creata
all_configs = load_configs(model_database_path)

# Salva il modello
model_name += ".pth"
save_model(
    model=vae_model,
    optimizer=optimizer,
    train_data=train_data,
    val_data=val_data,  # Aggiungi se hai un dataset di validazione
    epoch=50,
    stopped_epoch=stopped_epoch,
    encoder_layers=selected_model["encoder_layers"],
    decoder_layers=selected_model["decoder_layers"],
    train_losses=train_losses,
    val_losses=val_losses,
    recon_losses=recon_losses,
    kld_losses=kld_losses,
    beta_values=beta_values,
    gradient_history=gradient_history,
    output_folder="Models",
    model_name=model_name
)
